In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
DRIVE_DIR = '/content/drive/MyDrive/vit-from-scratch'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

In [ ]:
print(f"Drive folder ready: {DRIVE_DIR}")

Drive folder ready: /content/drive/MyDrive/vit-from-scratch


In [ ]:
%cd /content

/content


In [ ]:
!git clone https://github.com/parvpatodia/vit-from-scratch

Cloning into 'vit-from-scratch'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 39 (delta 4), reused 34 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 601.26 KiB | 3.32 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [ ]:
!ls /content/vit-from-scratch/models/

attention.py  mlp.py		  positional_encoding.py  transformer_block.py
__init__.py   patch_embedding.py  swin_attention.py	  vit.py


In [ ]:
%cd vit-from-scratch
!pip install -q -r requirements.txt

print("\nRepo cloned and dependencies installed!")

/content/vit-from-scratch

Repo cloned and dependencies installed!


In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name:        {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = torch.device('cuda')
else:
    print("⚠️  NO GPU DETECTED! Go to Runtime → Change runtime type → T4 GPU")
    device = torch.device('cpu')

print(f"\nUsing device: {device}")

PyTorch version: 2.10.0+cu128
CUDA available:  True
GPU name:        Tesla T4
GPU memory:      15.6 GB

Using device: cuda


In [ ]:
import os
import sys

# --- Repo root: directory that directly contains configs/, models/, utils/ ---
# Nested clone: ~/vit-from-scratch/vit-from-scratch. Single clone: ~/vit-from-scratch.
_INNER = os.path.expanduser("~/vit-from-scratch/vit-from-scratch")
_SINGLE = os.path.expanduser("~/vit-from-scratch")
REPO_ROOT = _INNER if os.path.isfile(os.path.join(_INNER, "utils", "dataset.py")) else _SINGLE

# Drop parent folder from path if it also has utils/ (old copy wins otherwise).
_OUTER = os.path.dirname(_INNER)
if os.path.isdir(os.path.join(_OUTER, "utils")):
    sys.path[:] = [p for p in sys.path if os.path.abspath(p) != os.path.abspath(_OUTER)]

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Drop cached utils* so a previously loaded wrong module is not reused
for _k in list(sys.modules):
    if _k == "utils" or _k.startswith("utils."):
        del sys.modules[_k]

import utils.dataset as _uds
assert hasattr(_uds, "get_cifar10_loaders"), "Wrong utils.dataset loaded; check REPO_ROOT and sys.path"
print("Using dataset.py from:", _uds.__file__)

from configs.config import ViTConfig, EuroSATConfig
from models.vit import ViT
from utils.dataset import get_cifar10_loaders, CIFAR10_CLASSES, denormalize_cifar10
from utils.training import train, load_checkpoint, evaluate
from utils.evaluation import full_evaluation
from utils.visualization import (
    plot_attention_maps,
    plot_attention_rollout,
    plot_attention_rollout_grid,
    plot_positional_embedding_similarity,
    plot_tsne_embeddings,
    plot_training_curves,
)

import matplotlib.pyplot as plt
import numpy as np

print("All imports successful!")

All imports successful!


In [ ]:
config = ViTConfig()

# Print config summary
print("=" * 50)
print("ViT Configuration")
print("=" * 50)
print(f"Image size:      {config.image_size}×{config.image_size}")
print(f"Patch size:      {config.patch_size}×{config.patch_size}")
print(f"Num patches:     {config.num_patches}")
print(f"d_model:         {config.d_model}")
print(f"Heads:           {config.num_heads}")
print(f"Layers:          {config.num_layers}")
print(f"FFN hidden:      {config.ffn_hidden}")
print(f"Total epochs:    {config.total_epochs}")
print(f"Batch size:      {config.batch_size}")
print(f"Learning rate:   {config.learning_rate}")

# Create model
model = ViT(config)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

# Create data loaders
train_loader, val_loader, test_loader = get_cifar10_loaders(config, num_workers=2)
print(f"\nTrain: {len(train_loader.dataset):,} images ({len(train_loader)} batches)")
print(f"Val:   {len(val_loader.dataset):,} images ({len(val_loader)} batches)")
print(f"Test:  {len(test_loader.dataset):,} images ({len(test_loader)} batches)")


ViT Configuration
Image size:      32×32
Patch size:      4×4
Num patches:     64
d_model:         128
Heads:           4
Layers:          6
FFN hidden:      512
Total epochs:    200
Batch size:      256
Learning rate:   0.001

Total parameters: 1,205,898


100%|██████████| 170M/170M [00:03<00:00, 48.2MB/s]



Train: 45,000 images (175 batches)
Val:   5,000 images (20 batches)
Test:  10,000 images (40 batches)


In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(16):
    row, col = i // 8, i % 8
    img = denormalize_cifar10(images[i]).permute(1, 2, 0).numpy()
    axes[row, col].imshow(img)
    axes[row, col].set_title(CIFAR10_CLASSES[labels[i]], fontsize=8)
    axes[row, col].axis('off')

plt.suptitle('Sample Training Images (with augmentation)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline

In [ ]:
SAVE_DIR = f'{DRIVE_DIR}/checkpoints'

history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    save_dir=SAVE_DIR,
    experiment_name="vit_cifar10"
)

print("\n Training complete!")

Training on: cuda
Parameters with weight decay:    1,187,072
Parameters without weight decay: 18,826

Epoch | Train Loss | Train Acc |   Val Loss |   Val Acc |         LR |   Time


    1 |     2.1984 |    17.77% |     2.0708 |    23.12% |   0.000010 |  39.7s


    2 |     2.0482 |    23.48% |     1.9096 |    29.56% |   0.000109 |  41.2s


    3 |     1.9293 |    28.70% |     1.7361 |    36.76% |   0.000208 |  41.5s


    4 |     1.8257 |    32.40% |     1.6191 |    41.94% |   0.000307 |  40.4s


...

  160 |     0.4837 |    82.90% |     0.5003 |    84.04% |   0.000119 |  39.1s


  165 |     0.4778 |    82.96% |     0.4941 |    84.88% |   0.000095 |  40.7s


  170 |     0.4665 |    83.50% |     0.4907 |    84.88% |   0.000074 |  40.6s


  175 |     0.4615 |    83.60% |     0.4862 |    85.20% |   0.000055 |  39.6s


Epoch 176 [Train]:  45%|████▍     | 78/175 [00:17<00:18,  5.20it/s, loss=0.3405, acc=83.8%]

In [ ]:

plot_training_curves(
    history,
    save_path=f'{DRIVE_DIR}/results/training_curves.png'
)

# Also show inline
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history['train_loss'], 'b-', label='Train', alpha=0.7)
ax1.plot(epochs, history['val_loss'], 'r-', label='Val', alpha=0.7)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history['train_acc'], 'b-', label='Train', alpha=0.7)
ax2.plot(epochs, history['val_acc'], 'r-', label='Val', alpha=0.7)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best val accuracy: {max(history['val_acc']):.2f}%")

In [ ]:
best_model = ViT(config).to(device)
checkpoint = load_checkpoint(
    best_model,
    f'{SAVE_DIR}/vit_cifar10_best.pt',
    device=device
)

# Full evaluation on test set
results = full_evaluation(
    best_model, test_loader, device,
    class_names=CIFAR10_CLASSES,
    save_dir=f'{DRIVE_DIR}/results',
    experiment_name='vit_cifar10'
)


In [ ]:
test_images, test_labels = next(iter(test_loader))

# Pick one image from each class for nice visualization
class_examples = {}
for img, label in zip(test_images, test_labels):
    cls = label.item()
    if cls not in class_examples:
        class_examples[cls] = img
    if len(class_examples) == 10:
        break

# Plot attention maps for one example
example_img = test_images[0:1]
example_label = CIFAR10_CLASSES[test_labels[0].item()]

plot_attention_maps(
    best_model, example_img, device,
    save_path=f'{DRIVE_DIR}/results/attention_maps_{example_label}.png'
)

# Show inline
from IPython.display import Image, display
display(Image(filename=f'{DRIVE_DIR}/results/attention_maps_{example_label}.png'))


In [ ]:
sample_images = test_images[:8]
sample_labels = test_labels[:8]

plot_attention_rollout_grid(
    best_model, sample_images, sample_labels, device,
    class_names=CIFAR10_CLASSES,
    save_path=f'{DRIVE_DIR}/results/attention_rollout_grid.png'
)

display(Image(filename=f'{DRIVE_DIR}/results/attention_rollout_grid.png'))

In [ ]:

fig, axes = plt.subplots(2, 10, figsize=(25, 5))

for cls_idx in range(10):
    if cls_idx in class_examples:
        img = class_examples[cls_idx].unsqueeze(0)

        # Get prediction
        with torch.no_grad():
            logit = best_model(img.to(device))
            pred = logit.argmax(1).item()

        # Original image
        img_display = denormalize_cifar10(img[0].cpu()).permute(1, 2, 0).numpy()
        axes[0, cls_idx].imshow(img_display)
        color = 'green' if pred == cls_idx else 'red'
        axes[0, cls_idx].set_title(f'{CIFAR10_CLASSES[cls_idx]}', fontsize=9, color=color)
        axes[0, cls_idx].axis('off')

        # Attention rollout
        from utils.visualization import compute_attention_rollout
        rollout = compute_attention_rollout(best_model, img, device)
        rollout_resized = torch.nn.functional.interpolate(
            torch.tensor(rollout).unsqueeze(0).unsqueeze(0),
            size=(config.image_size, config.image_size),
            mode='bilinear', align_corners=False
        ).squeeze().numpy()

        axes[1, cls_idx].imshow(img_display)
        axes[1, cls_idx].imshow(rollout_resized, alpha=0.6, cmap='viridis')
        axes[1, cls_idx].axis('off')

plt.suptitle('Attention Rollout: One Example Per Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/results/rollout_per_class.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:

plot_positional_embedding_similarity(
    best_model,
    save_path=f'{DRIVE_DIR}/results/pos_embed_similarity.png'
)

display(Image(filename=f'{DRIVE_DIR}/results/pos_embed_similarity.png'))

In [ ]:

plot_tsne_embeddings(
    best_model, test_loader, device,
    class_names=CIFAR10_CLASSES,
    max_samples=3000,
    save_path=f'{DRIVE_DIR}/results/tsne_embeddings.png'
)

display(Image(filename=f'{DRIVE_DIR}/results/tsne_embeddings.png'))

In [ ]:
import torch
torch.save(history, f'{DRIVE_DIR}/checkpoints/vit_cifar10_history.pt')

# Print summary
print("=" * 60)
print("TRAINING COMPLETE — SUMMARY")
print("=" * 60)
print(f"Best validation accuracy:  {max(history['val_acc']):.2f}%")
print(f"Test Top-1 accuracy:       {results['top1_accuracy']:.2f}%")
print(f"Test Top-5 accuracy:       {results['top5_accuracy']:.2f}%")
print(f"\nAll files saved to Google Drive:")
print(f"  {DRIVE_DIR}/checkpoints/vit_cifar10_best.pt")
print(f"  {DRIVE_DIR}/checkpoints/vit_cifar10_history.pt")
print(f"  {DRIVE_DIR}/results/training_curves.png")
print(f"  {DRIVE_DIR}/results/vit_cifar10_confusion_matrix.png")
print(f"  {DRIVE_DIR}/results/attention_maps_*.png")
print(f"  {DRIVE_DIR}/results/attention_rollout_grid.png")
print(f"  {DRIVE_DIR}/results/rollout_per_class.png")
print(f"  {DRIVE_DIR}/results/pos_embed_similarity.png")
print(f"  {DRIVE_DIR}/results/tsne_embeddings.png")

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/vit-from-scratch'
ABLATION_DIR = f'{DRIVE_DIR}/ablation_results'
os.makedirs(ABLATION_DIR, exist_ok=True)

%cd /content
!rm -rf vit-from-scratch
!git clone https://github.com/parvpatodia/vit-from-scratch.git
%cd vit-from-scratch
!pip install -q -r requirements.txt

%matplotlib inline

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import sys

_INNER = os.path.expanduser("~/vit-from-scratch/vit-from-scratch")
_SINGLE = os.path.expanduser("~/vit-from-scratch")
REPO_ROOT = _INNER if os.path.isfile(os.path.join(_INNER, "utils", "dataset.py")) else _SINGLE
_OUTER = os.path.dirname(_INNER)
if os.path.isdir(os.path.join(_OUTER, "utils")):
    sys.path[:] = [p for p in sys.path if os.path.abspath(p) != os.path.abspath(_OUTER)]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
for _k in list(sys.modules):
    if _k == "utils" or _k.startswith("utils."):
        del sys.modules[_k]

import time
import math
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from dataclasses import replace

from configs.config import ViTConfig
from models.vit import ViT
from utils.dataset import get_cifar10_loaders, CIFAR10_CLASSES
from utils.training import train, evaluate, load_checkpoint

# Create data loaders ONCE (shared across all ablations)
base_config = ViTConfig()
train_loader, val_loader, test_loader = get_cifar10_loaders(base_config, num_workers=2)

print(f"Data loaded: {len(train_loader.dataset)} train, {len(val_loader.dataset)} val, {len(test_loader.dataset)} test")


In [ ]:

ABLATION_EPOCHS = 50  # Enough to see clear differences (~35 min each on T4)

def run_ablation(
    experiment_name: str,
    config: ViTConfig = None,
    use_cls_token: bool = True,
    use_scaling: bool = True,
    block_type: str = "pre_norm",
    description: str = ""
):
    """
    Run a single ablation experiment.

    Returns:
        dict with history and final metrics
    """
    if config is None:
        config = ViTConfig()

    # Override epochs for ablation
    config.total_epochs = ABLATION_EPOCHS
    config.warmup_epochs = 5  # Shorter warmup for shorter training

    print("\n" + "=" * 70)
    print(f"ABLATION: {experiment_name}")
    print(f"  {description}")
    print("=" * 70)

    # Create model with ablation settings
    model = ViT(
        config,
        use_cls_token=use_cls_token,
        use_scaling=use_scaling,
        block_type=block_type
    )

    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {total_params:,}")

    # Train
    history = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        save_dir=ABLATION_DIR,
        experiment_name=experiment_name
    )

    # Evaluate on test set
    best_model = ViT(
        config,
        use_cls_token=use_cls_token,
        use_scaling=use_scaling,
        block_type=block_type
    ).to(device)
    load_checkpoint(best_model, f'{ABLATION_DIR}/{experiment_name}_best.pt', device=device)
    test_loss, test_acc = evaluate(best_model, test_loader, device, desc="Test")

    result = {
        'name': experiment_name,
        'description': description,
        'history': history,
        'best_val_acc': max(history['val_acc']),
        'test_acc': test_acc,
        'test_loss': test_loss,
        'params': total_params,
    }

    print(f"\n  Best val acc:  {result['best_val_acc']:.2f}%")
    print(f"  Test acc:      {result['test_acc']:.2f}%")

    return result

# Store all results
all_results = {}
print("Ablation framework ready!")


In [ ]:

all_results['baseline'] = run_ablation(
    experiment_name="baseline_50ep",
    description="Standard ViT with default config (50 epochs)"
)


In [ ]:

# HYPOTHESIS:
# Without scaling, dot products Q·K have variance = d_k = 32.
# This means softmax inputs have std ≈ 5.7 instead of ≈ 1.
# Softmax will saturate → attention becomes nearly one-hot →
# gradients vanish → model can't learn effectively.
#
# EXPECTED: Significant accuracy drop, attention entropy collapse.

all_results['no_scaling'] = run_ablation(
    experiment_name="no_scaling",
    use_scaling=False,
    description="Remove √d_k scaling from attention scores"
)


In [ ]:

# HYPOTHESIS:
# Without positional encoding, self-attention is permutation
# equivariant — the model treats patches as an unordered set.
# It can still learn color/texture features but loses ALL
# spatial information. Classes that depend on spatial arrangement
# (like 'ship' vs 'airplane') should suffer most.
#
# EXPECTED: Noticeable accuracy drop, attention maps lose
# spatial coherence.

# Create a ViT with zeroed-out positional embeddings
class ViTNoPosition(ViT):
    """ViT with positional embeddings forced to zero."""
    def __init__(self, config):
        super().__init__(config)
        # Zero out positional embeddings and freeze them
        with torch.no_grad():
            self.patch_embed.pos_embed.zero_()
        self.patch_embed.pos_embed.requires_grad = False

# Custom training for this variant
config_nopos = ViTConfig()
config_nopos.total_epochs = ABLATION_EPOCHS
config_nopos.warmup_epochs = 5

print("\n" + "=" * 70)
print("ABLATION: No Positional Encoding")
print("  Remove all positional information from the model")
print("=" * 70)

model_nopos = ViTNoPosition(config_nopos)
print(f"  Parameters: {sum(p.numel() for p in model_nopos.parameters()):,}")
print(f"  Trainable:  {sum(p.numel() for p in model_nopos.parameters() if p.requires_grad):,}")
print(f"  (Positional embeddings frozen at zero)")

history_nopos = train(
    model=model_nopos,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config_nopos,
    device=device,
    save_dir=ABLATION_DIR,
    experiment_name="no_position"
)

test_loss_nopos, test_acc_nopos = evaluate(model_nopos, test_loader, device, desc="Test")

all_results['no_position'] = {
    'name': 'no_position',
    'description': 'No positional encoding (zeroed and frozen)',
    'history': history_nopos,
    'best_val_acc': max(history_nopos['val_acc']),
    'test_acc': test_acc_nopos,
    'test_loss': test_loss_nopos,
    'params': sum(p.numel() for p in model_nopos.parameters()),
}

print(f"\n  Best val acc:  {all_results['no_position']['best_val_acc']:.2f}%")
print(f"  Test acc:      {test_acc_nopos:.2f}%")


In [ ]:
# HYPOTHESIS:
# Smaller patches → more tokens → finer detail but O(N²) cost.
# Larger patches → fewer tokens → coarser but much faster.
#
# patch_size=2: 256 patches (very slow, very detailed)
# patch_size=4: 64 patches (default, good balance)
# patch_size=8: 16 patches (fast but coarse)
# patch_size=16: 4 patches (too coarse, should fail badly)
#
# EXPECTED: Accuracy: P2 ≥ P4 > P8 >> P16
#           Speed: P16 >> P8 > P4 >> P2

for patch_size in [2, 8, 16]:
    config_ps = ViTConfig()
    config_ps.patch_size = patch_size

    # Adjust num_heads if d_model / num_heads would be too small
    # (for very small models with patch_size=16, fewer params)

    all_results[f'patch_{patch_size}'] = run_ablation(
        experiment_name=f"patch_size_{patch_size}",
        config=config_ps,
        description=f"Patch size = {patch_size} ({(config_ps.image_size // patch_size) ** 2} patches)"
    )



In [ ]:

# HYPOTHESIS:
# More heads = more diverse attention patterns, but each head
# has smaller dimension (d_k = d_model / num_heads).
# 1 head: d_k=128 (big head, one attention pattern)
# 2 heads: d_k=64
# 4 heads: d_k=32 (default)
# 8 heads: d_k=16 (many patterns, but each head is small)
#
# EXPECTED: Sweet spot around 4 heads. 1 head = limited diversity.
# 8 heads = each head might be too small to capture complex patterns.

for num_heads in [1, 2, 8]:
    config_nh = ViTConfig()
    config_nh.num_heads = num_heads

    all_results[f'heads_{num_heads}'] = run_ablation(
        experiment_name=f"heads_{num_heads}",
        config=config_nh,
        description=f"{num_heads} attention head(s), d_k = {config_nh.d_model // num_heads}"
    )


In [ ]:

# HYPOTHESIS:
# Without residual connections, gradients must flow through
# all 6 layers sequentially. Each layer's Jacobian is multiplied,
# causing exponential decay (vanishing gradients).
# The model should fail to train or train very poorly.
#
# EXPECTED: Dramatic accuracy drop, possibly no learning at all.
# Gradient norms in early layers should be near zero.

all_results['no_residual'] = run_ablation(
    experiment_name="no_residual",
    block_type="no_residual",
    description="Remove all residual/skip connections"
)



In [ ]:
# HYPOTHESIS:
# Post-Norm puts LayerNorm after the residual addition:
#   output = LayerNorm(x + Sublayer(x))
# This means the gradient through the skip connection must
# pass through LayerNorm's Jacobian, which can distort it.
#
# Pre-Norm (our default) keeps the skip connection clean:
#   output = x + Sublayer(LayerNorm(x))
#
# EXPECTED: Post-Norm should train less stably, possibly lower
# accuracy. Gradient norms should be less uniform across layers.

all_results['post_norm'] = run_ablation(
    experiment_name="post_norm",
    block_type="post_norm",
    description="Post-Norm: LayerNorm(x + Attention(x)) instead of x + Attention(LN(x))"
)


In [ ]:

# HYPOTHESIS:
# [CLS] token: a learnable token that aggregates info via attention.
#   - More expressive (dedicated aggregation token)
#   - Has extra parameters (the CLS token itself)
#
# Global Average Pooling (GAP): average all patch token outputs.
#   - No extra parameters
#   - Every patch contributes equally
#
# EXPECTED: Similar accuracy. GAP might be slightly better on
# small datasets because it doesn't need to learn an aggregation
# strategy from scratch.

all_results['gap'] = run_ablation(
    experiment_name="global_avg_pool",
    use_cls_token=False,
    description="Global Average Pooling instead of [CLS] token"
)


In [ ]:

print("\n" + "=" * 90)
print("ABLATION STUDY RESULTS")
print("=" * 90)
print(f"{'Experiment':<30s} | {'Val Acc':>8s} | {'Test Acc':>8s} | {'Δ vs Base':>9s} | {'Params':>10s}")
print("-" * 90)

baseline_acc = all_results['baseline']['best_val_acc']

# Sort by val accuracy descending
sorted_results = sorted(all_results.values(), key=lambda x: x['best_val_acc'], reverse=True)

for r in sorted_results:
    delta = r['best_val_acc'] - baseline_acc
    delta_str = f"{delta:+.2f}%" if r['name'] != 'baseline_50ep' else "—"
    print(f"{r['description'][:30]:<30s} | {r['best_val_acc']:>7.2f}% | {r['test_acc']:>7.2f}% | {delta_str:>9s} | {r['params']:>10,}")

print("=" * 90)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Define colors for each ablation
colors = {
    'baseline_50ep': 'black',
    'no_scaling': 'red',
    'no_position': 'orange',
    'patch_size_2': 'cyan',
    'patch_size_8': 'blue',
    'patch_size_16': 'navy',
    'heads_1': 'green',
    'heads_2': 'lime',
    'heads_8': 'darkgreen',
    'no_residual': 'magenta',
    'post_norm': 'purple',
    'global_avg_pool': 'brown',
}

for name, result in all_results.items():
    h = result['history']
    epochs = range(1, len(h['val_acc']) + 1)
    color = colors.get(name, 'gray')
    label = result['description'][:35]

    axes[0].plot(epochs, h['train_loss'], color=color, alpha=0.7, label=label)
    axes[1].plot(epochs, h['val_acc'], color=color, alpha=0.7, label=label)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss — All Ablations')
axes[0].legend(fontsize=7, loc='upper right')
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Accuracy (%)')
axes[1].set_title('Validation Accuracy — All Ablations')
axes[1].legend(fontsize=7, loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{ABLATION_DIR}/all_ablations_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved to: {ABLATION_DIR}/all_ablations_curves.png")



In [ ]:

fig, ax = plt.subplots(figsize=(14, 6))

names = [r['description'][:25] for r in sorted_results]
val_accs = [r['best_val_acc'] for r in sorted_results]
test_accs = [r['test_acc'] for r in sorted_results]

x = np.arange(len(names))
width = 0.35

bars1 = ax.bar(x - width/2, val_accs, width, label='Val Accuracy', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, test_accs, width, label='Test Accuracy', color='coral', alpha=0.8)

# Add baseline line
ax.axhline(y=baseline_acc, color='black', linestyle='--', alpha=0.5, label=f'Baseline ({baseline_acc:.1f}%)')

ax.set_ylabel('Accuracy (%)')
ax.set_title('Ablation Study: Accuracy Comparison')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(f'{ABLATION_DIR}/ablation_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
 # This directly validates the √d_k derivation.

from utils.visualization import compute_attention_rollout

# Load both models
model_baseline = ViT(ViTConfig()).to(device)
load_checkpoint(model_baseline, f'{ABLATION_DIR}/baseline_50ep_best.pt', device=device)

config_noscale = ViTConfig()
model_noscale = ViT(config_noscale, use_scaling=False).to(device)
load_checkpoint(model_noscale, f'{ABLATION_DIR}/no_scaling_best.pt', device=device)

# Get a batch of test images
test_images, test_labels = next(iter(test_loader))

# Compute attention maps for both
model_baseline.eval()
model_noscale.eval()

with torch.no_grad():
    attn_baseline = model_baseline.get_attention_maps(test_images[:16].to(device)).cpu()
    attn_noscale = model_noscale.get_attention_maps(test_images[:16].to(device)).cpu()

# Compute entropy for each
def attention_entropy(attn_maps):
    """Compute mean attention entropy across all layers, heads, and tokens."""
    # attn_maps: [layers, batch, heads, N, N]
    # Entropy per row: -Σ p * log(p)
    entropy = -(attn_maps * (attn_maps + 1e-10).log()).sum(dim=-1)
    return entropy.mean(dim=(1, 2, 3))  # Mean over batch, heads, tokens → per layer

entropy_baseline = attention_entropy(attn_baseline)
entropy_noscale = attention_entropy(attn_noscale)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-layer entropy comparison
layers = range(1, len(entropy_baseline) + 1)
axes[0].bar(np.array(list(layers)) - 0.15, entropy_baseline.numpy(), 0.3,
            label='With √d_k scaling', color='steelblue', alpha=0.8)
axes[0].bar(np.array(list(layers)) + 0.15, entropy_noscale.numpy(), 0.3,
            label='Without scaling', color='coral', alpha=0.8)
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Attention Entropy')
axes[0].set_title('Attention Entropy per Layer\n(Higher = more uniform attention)')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

# Attention weight distribution for one layer
layer_idx = 0  # First layer
attn_base_flat = attn_baseline[layer_idx, 0, 0, 0, 1:].numpy()  # CLS → patches
attn_noscale_flat = attn_noscale[layer_idx, 0, 0, 0, 1:].numpy()

axes[1].bar(range(len(attn_base_flat)), attn_base_flat, alpha=0.6,
            label='With √d_k scaling', color='steelblue')
axes[1].bar(range(len(attn_noscale_flat)), attn_noscale_flat, alpha=0.6,
            label='Without scaling', color='coral')
axes[1].set_xlabel('Patch Index')
axes[1].set_ylabel('Attention Weight')
axes[1].set_title(f'[CLS] Attention Distribution (Layer 1, Head 1)\n(Without scaling → nearly one-hot)')
axes[1].legend()
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{ABLATION_DIR}/scaling_entropy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMean entropy WITH scaling:    {entropy_baseline.mean():.4f}")
print(f"Mean entropy WITHOUT scaling: {entropy_noscale.mean():.4f}")
print(f"Ratio: {entropy_baseline.mean() / entropy_noscale.mean():.1f}× higher entropy with scaling")


In [ ]:
#  Compare attention patterns between baseline and no-position model

from utils.dataset import denormalize_cifar10

model_nopos_loaded = ViTNoPosition(ViTConfig()).to(device)
load_checkpoint(model_nopos_loaded, f'{ABLATION_DIR}/no_position_best.pt', device=device)

# Pick one test image
img = test_images[0:1].to(device)
img_display = denormalize_cifar10(test_images[0]).permute(1, 2, 0).numpy()

with torch.no_grad():
    rollout_baseline = compute_attention_rollout(model_baseline, img, device)
    rollout_nopos = compute_attention_rollout(model_nopos_loaded, img, device)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

config = ViTConfig()

# Resize rollouts
def resize_rollout(rollout, size):
    return F.interpolate(
        torch.tensor(rollout).unsqueeze(0).unsqueeze(0),
        size=(size, size), mode='bilinear', align_corners=False
    ).squeeze().numpy()

rollout_base_resized = resize_rollout(rollout_baseline, config.image_size)
rollout_nopos_resized = resize_rollout(rollout_nopos, config.image_size)

axes[0].imshow(img_display)
axes[0].set_title('Original Image', fontsize=11)
axes[0].axis('off')

axes[1].imshow(img_display)
axes[1].imshow(rollout_base_resized, alpha=0.6, cmap='viridis')
axes[1].set_title('Attention WITH Positions\n(spatially focused)', fontsize=11)
axes[1].axis('off')

axes[2].imshow(img_display)
axes[2].imshow(rollout_nopos_resized, alpha=0.6, cmap='viridis')
axes[2].set_title('Attention WITHOUT Positions\n(spatially unfocused)', fontsize=11)
axes[2].axis('off')

plt.suptitle('Effect of Positional Encoding on Attention', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{ABLATION_DIR}/position_encoding_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

patch_sizes = [2, 4, 8, 16]
num_patches_list = [(32 // p) ** 2 for p in patch_sizes]
attention_flops = [n * n * 128 for n in num_patches_list]  # O(N² × d)

# Get training times from results
times = {}
for ps in patch_sizes:
    key = f'patch_{ps}' if ps != 4 else 'baseline'
    if key in all_results:
        times[ps] = sum(all_results[key]['history']['epoch_time']) / len(all_results[key]['history']['epoch_time'])
    else:
        times[ps] = None

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Num patches vs patch size
axes[0].bar([str(p) for p in patch_sizes], num_patches_list, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Patch Size')
axes[0].set_ylabel('Number of Patches')
axes[0].set_title('Patches vs Patch Size')
axes[0].grid(True, axis='y', alpha=0.3)
for i, (ps, np_) in enumerate(zip(patch_sizes, num_patches_list)):
    axes[0].annotate(f'{np_}', (i, np_), ha='center', va='bottom', fontsize=10)

# Attention FLOPs
axes[1].bar([str(p) for p in patch_sizes], [f/1e6 for f in attention_flops], color='coral', alpha=0.8)
axes[1].set_xlabel('Patch Size')
axes[1].set_ylabel('Attention FLOPs (millions)')
axes[1].set_title('Attention Computation Cost (O(N²·d))')
axes[1].set_yscale('log')
axes[1].grid(True, axis='y', alpha=0.3)

# Accuracy vs patch size
accs = []
for ps in patch_sizes:
    key = f'patch_{ps}' if ps != 4 else 'baseline'
    if key in all_results:
        accs.append(all_results[key]['best_val_acc'])
    else:
        accs.append(None)

valid_ps = [ps for ps, a in zip(patch_sizes, accs) if a is not None]
valid_accs = [a for a in accs if a is not None]
axes[2].plot(valid_ps, valid_accs, 'bo-', markersize=8, linewidth=2)
axes[2].set_xlabel('Patch Size')
axes[2].set_ylabel('Best Val Accuracy (%)')
axes[2].set_title('Accuracy vs Patch Size')
axes[2].grid(True, alpha=0.3)
for ps, acc in zip(valid_ps, valid_accs):
    axes[2].annotate(f'{acc:.1f}%', (ps, acc), textcoords="offset points",
                     xytext=(0, 10), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(f'{ABLATION_DIR}/patch_size_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:

# Save the complete results dict
torch.save(all_results, f'{ABLATION_DIR}/all_ablation_results.pt')

print("=" * 70)
print("ALL ABLATION STUDIES COMPLETE")
print("=" * 70)
print(f"\nResults saved to: {ABLATION_DIR}/")
print(f"\nFiles generated:")
print(f"  - all_ablation_results.pt (raw data)")
print(f"  - all_ablations_curves.png")
print(f"  - ablation_bar_chart.png")
print(f"  - scaling_entropy_analysis.png")
print(f"  - position_encoding_comparison.png")
print(f"  - patch_size_analysis.png")
print(f"  - Individual checkpoints for each ablation")
print(f"\nKey findings:")

for r in sorted_results:
    delta = r['best_val_acc'] - baseline_acc
    symbol = "✅" if abs(delta) < 2 else ("📉" if delta < -2 else "📈")
    print(f"  {symbol} {r['description'][:40]:<40s}: {r['best_val_acc']:.2f}% ({delta:+.2f}%)")